In [1]:
#Exercise-9!/usr/bin/env python3
"""
Hopfield Network Associative Memory & Pattern Reconstruction (handwritten digits)
--------------------------------------------------------------------------------
Dataset : Optical Recognition of Handwritten Digits (UCI optdigits, 8x8, values 0-16)

Run     : python hopfield_digits.py [optdigits.tra | optdigits.tes]

Data lookup order: file argument -> optdigits.tra/.tes in folder ->
                   scikit-learn's bundled UCI optdigits copy (offline) -> UCI download.
Pixels >= 8 become +1 (active), others -1 (inactive).  64 neurons.
Learning  : Hebb rule  W = (1/N) sum_p  x_p x_p^T ,  zero diagonal
Update    : asynchronous, random order, until no neuron changes
Energy    : E = -1/2 x^T W x

Requires: numpy   (scikit-learn optional, used as an offline data source)
"""
import os
import sys
import urllib.request

import numpy as np

SEED = 5
DEFAULT_DIGITS = [2, 6, 7]     # weakly correlated digits: plain Hebb stores these reliably
N = 64
THRESHOLD = 8
UCI_BASE = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/"


# ----------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------
def _csv(path):
    a = np.loadtxt(path, delimiter=",")
    return a[:, :64], a[:, 64].astype(int)


def load_data(path=None):
    for p in ([path] if path else []) + ["optdigits.tra", "optdigits.tes"]:
        if p and os.path.exists(p):
            print(f"[data] Loading {p}")
            X, y = _csv(p); return X, y, "file: " + p
    try:
        from sklearn.datasets import load_digits
        d = load_digits()
        print("[data] Using scikit-learn bundled UCI optdigits (1797 samples).")
        return d.data.astype(float), d.target.astype(int), "UCI optdigits via scikit-learn"
    except Exception:
        pass
    try:
        print("[data] Downloading optdigits.tra from UCI ...")
        urllib.request.urlretrieve(UCI_BASE + "optdigits.tra", "optdigits.tra")
        X, y = _csv("optdigits.tra"); return X, y, "UCI download"
    except Exception as e:
        sys.exit(f"[data] Could not load a dataset ({e}). Put optdigits.tra next to this script.")


def to_bipolar(X): return np.where(X >= THRESHOLD, 1, -1).astype(int)


def show(p, title=""):
    if title: print(title)
    for r in p.reshape(8, 8):
        print("   " + " ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r))


def row(p):
    return [" ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r) for r in p.reshape(8, 8)]


def show_many(pats, titles, gap="     "):
    print("   " + gap.join(f"{t:<15}" for t in titles))
    rows = [row(p) for p in pats]
    for r in range(8):
        print("   " + gap.join(f"{rw[r]:<15}" for rw in rows))


# ----------------------------------------------------------------------------
# Hopfield network
# ----------------------------------------------------------------------------
class Hopfield:
    def __init__(self):
        self.W = np.zeros((N, N)); self.P = np.zeros((0, N), int); self.labels = []; self.rule = "hebb"

    def store(self, patterns, labels, rule="hebb"):
        """rule 'hebb': W = (1/N) sum x x^T   |   'pinv': projection (pseudo-inverse) rule, for comparison"""
        self.P = np.array(patterns, int); self.labels = list(labels); self.rule = rule
        if rule == "hebb":
            self.W = sum(np.outer(p, p) for p in self.P).astype(float) / N
        else:
            self.W = self.P.T @ np.linalg.pinv(self.P @ self.P.T) @ self.P
        np.fill_diagonal(self.W, 0)

    def energy(self, x): return float(-0.5 * x @ self.W @ x)

    def recall(self, x, rng, max_sweeps=50):
        """Asynchronous recall. x may contain 0 (unknown). Returns state, sweeps, energy trace."""
        x = x.astype(int).copy()
        trace = [self.energy(x)]
        for sw in range(1, max_sweeps + 1):
            changed = 0
            for i in rng.permutation(N):
                h = self.W[i] @ x
                new = x[i] if h == 0 and x[i] != 0 else (1 if h >= 0 else -1)
                if new != x[i]: x[i] = new; changed += 1
            trace.append(self.energy(x))
            if changed == 0: return x, sw, trace
        return x, max_sweeps, trace

    def identify(self, x):
        """Classify a fixed point: ('stored', label) | ('mirror', label) | ('spurious', nearest_label, hamming)."""
        for p, l in zip(self.P, self.labels):
            if np.array_equal(x, p): return "stored", l
        for p, l in zip(self.P, self.labels):
            if np.array_equal(x, -p): return "mirror", l
        d = [(int((x != p).sum()), l) for p, l in zip(self.P, self.labels)]
        h, l = min(d)
        return "spurious", l, h


# ----------------------------------------------------------------------------
# application
# ----------------------------------------------------------------------------
def parse_digits(s, default):
    try:
        v = sorted({int(t) for t in s.replace(",", " ").split() if 0 <= int(t) <= 9})
        return v or default
    except ValueError:
        return default


class App:
    def __init__(self, X, y, source):
        self.source = source
        self.rng = np.random.default_rng(SEED)
        self.Xb, self.y = to_bipolar(X), y
        self.net = Hopfield()
        self.proto = {d: np.where(self.Xb[y == d].mean(0) > 0, 1, -1) for d in range(10)}
        self.digits = []
        self.last = {}
        self.store_digits(DEFAULT_DIGITS, quiet=True)

    def store_digits(self, digits, quiet=False, mode="prototype", rule="hebb"):
        self.digits = list(digits)
        pats = []
        for d in self.digits:
            if mode == "prototype": pats.append(self.proto[d])
            else:
                idx = np.where(self.y == d)[0]; pats.append(self.Xb[self.rng.choice(idx)])
        self.net.store(pats, self.digits, rule)
        self.mode = mode; self.rule = rule

    def need(self):
        if len(self.digits) == 0: self.op1()

    def stable_count(self):
        return sum(np.array_equal(np.where(self.net.W @ p >= 0, 1, -1), p) for p in self.net.P)

    def noisy(self, x, frac):
        x = x.copy(); k = int(round(frac * N))
        x[self.rng.choice(N, k, replace=False)] *= -1
        return x

    def pick_digit(self):
        while True:
            s = input(f"Stored digit to use {self.digits}: ").strip()
            if s.isdigit() and int(s) in self.digits: return int(s)
            if s == "": return int(self.rng.choice(self.digits))
            print("  that digit is not stored.")

    # -- 1
    def op1(self):
        s = input("Digits to store (e.g. '2 6 7' or '0 1 2 3 4 5 6 7 8 9', default 2 6 7): ").strip()
        digits = parse_digits(s, DEFAULT_DIGITS)
        m = input("Pattern source: 1) class-average prototype (default)  2) one random sample each: ").strip()
        r = input("Learning rule: 1) Hebbian (default)  2) pseudo-inverse (comparison, handles correlated digits): ").strip()
        self.store_digits(digits, mode="sample" if m == "2" else "prototype", rule="pinv" if r == "2" else "hebb")
        rn = "Hebb rule  W = (1/%d) sum x x^T" % N if self.rule == "hebb" else "pseudo-inverse rule  W = P^T (P P^T)^-1 P"
        print(f"\nStored {len(digits)} patterns by the {rn}  ({self.mode})")
        for d, p in zip(self.digits, self.net.P):
            show(p, f"\nDigit {d}  (energy {self.net.energy(p):.3f})")
        print(f"\nTheoretical capacity of {N} neurons ~ 0.138*N = {0.138*N:.1f} random patterns")
        st = self.stable_count()
        print(f"Stored patterns that are stable fixed points: {st}/{len(digits)}")
        if st < len(digits) and self.rule == "hebb":
            print("-> Digit images are highly correlated, so plain Hebbian crosstalk destabilises some memories."
                  "\n   Store fewer / less similar digits (e.g. 2 6 7) or compare with the pseudo-inverse rule.")
        c = np.corrcoef(self.net.P)
        off = np.abs(c[~np.eye(len(digits), dtype=bool)]) if len(digits) > 1 else np.array([0])
        print(f"Mean |correlation| between stored patterns: {off.mean():.2f} "
              f"(high correlation lowers real capacity)")

    # -- 2
    def op2(self):
        self.need()
        d = self.pick_digit()
        f = float(input("Noise fraction (0-1, default 0.2): ") or 0.2)
        true = self.net.P[self.digits.index(d)]
        x0 = self.noisy(true, f)
        x, sw, tr = self.net.recall(x0, self.rng)
        show_many([x0, x, true], ["Noisy input", "Recalled", "Original"])
        res = self.net.identify(x)
        print(f"\nPixels corrupted: {(x0 != true).sum()}   remaining wrong: {(x != true).sum()}   sweeps: {sw}")
        print("Energy per sweep: " + " -> ".join(f"{e:.2f}" for e in tr))
        self.report_fp(res, d)

    def report_fp(self, res, d):
        if res[0] == "stored":
            print(f"Result: converged to stored digit {res[1]}  -> {'CORRECT' if res[1]==d else 'WRONG MEMORY'}")
        elif res[0] == "mirror":
            print(f"Result: converged to the INVERSE (mirror image) of digit {res[1]} - a false memory")
        else:
            print(f"Result: SPURIOUS state (closest to digit {res[1]}, {res[2]} pixels away) - a false memory")

    # -- 3
    def op3(self):
        self.need()
        print("Missing region: 1) top half 2) bottom half 3) left half 4) right half 5) random pixels")
        m = int(input("Mode (default 2): ") or 2)
        keep = np.ones((8, 8), bool)
        if m == 1: keep[:4] = False
        elif m == 2: keep[4:] = False
        elif m == 3: keep[:, :4] = False
        elif m == 4: keep[:, 4:] = False
        else:
            f = float(input("Fraction of pixels missing (default 0.5): ") or 0.5)
            keep = (self.rng.permutation(64) >= int(f * 64)).reshape(8, 8)
        keep = keep.ravel()
        d = self.pick_digit(); true = self.net.P[self.digits.index(d)]
        x0 = np.where(keep, true, 0)
        x, sw, tr = self.net.recall(x0, self.rng)
        show_many([x0, x, true], ["Input (blank=?)", "Reconstructed", "Original"])
        hid = ~keep
        print(f"\nMissing pixels: {hid.sum()}   restored correctly: {(x[hid]==true[hid]).sum()} "
              f"({(x[hid]==true[hid]).mean()*100:.1f}%)   sweeps: {sw}")
        self.report_fp(self.net.identify(x), d)
        ok = 0
        for dd, p in zip(self.digits, self.net.P):
            xx, _, _ = self.net.recall(np.where(keep, p, 0), self.rng)
            ok += np.array_equal(xx, p)
        print(f"Same mask on every stored digit: {ok}/{len(self.digits)} perfectly reconstructed")

    # -- 4
    def op4(self):
        self.need()
        trials = int(input("Trials per digit and noise level (default 30): ") or 30)
        levels = [0, .05, .1, .15, .2, .25, .3, .4, .5]
        print(f"\nPerfect-recall rate (%) per stored digit:")
        print(f"{'noise':>7}" + "".join(f"{d:>7}" for d in self.digits) + f"{'mean':>8}")
        for f in levels:
            rates = []
            for p in self.net.P:
                ok = sum(np.array_equal(self.net.recall(self.noisy(p, f), self.rng)[0], p) for _ in range(trials))
                rates.append(ok / trials * 100)
            print(f"{f*100:>6.0f}%" + "".join(f"{r:>7.0f}" for r in rates) + f"{np.mean(rates):>8.1f}")
        print("\nTolerance = highest noise level with mean recall >= 90% (see table). "
              "Above ~25-30% the input is closer to another attractor, or to the mirror image.")

    # -- 5
    def op5(self):
        trials = int(input("Trials per size (default 20): ") or 20)
        print("\nA) Digit patterns (random real samples of all digits stored together)")
        print(f"{'stored':>7}{'stable %':>10}{'recall@10% noise':>18}{'recall@20% noise':>18}{'pinv stable %':>15}")
        for m in (1, 2, 3, 4, 5, 6, 8, 10, 15, 20):
            sw, r10, r20, pv = [], [], [], []
            for _ in range(trials):
                idx = self.rng.choice(len(self.y), m, replace=False)
                netp = Hopfield()
                if len({tuple(r) for r in self.Xb[idx]}) == m:
                    netp.store(self.Xb[idx], self.y[idx], "pinv")
                    pv.append(np.mean([np.array_equal(np.where(netp.W @ p >= 0, 1, -1), p) for p in netp.P]))
                net = Hopfield(); net.store(self.Xb[idx], self.y[idx])
                sw.append(np.mean([np.array_equal(np.where(net.W @ p >= 0, 1, -1), p) for p in net.P]))
                p = net.P[self.rng.integers(m)]
                r10.append(np.array_equal(net.recall(self.noisy(p, .1), self.rng)[0], p))
                r20.append(np.array_equal(net.recall(self.noisy(p, .2), self.rng)[0], p))
            print(f"{m:>7}{np.mean(sw)*100:>9.0f}%{np.mean(r10)*100:>17.0f}%{np.mean(r20)*100:>17.0f}%{np.mean(pv)*100 if pv else float('nan'):>14.0f}%")
        print("\nB) Random uncorrelated patterns (theory: capacity ~ 0.138 N = 8.8)")
        print(f"{'stored':>7}{'stable %':>10}")
        for m in (2, 4, 6, 8, 10, 14, 20):
            sw = []
            for _ in range(trials):
                P = self.rng.choice([-1, 1], (m, N)); net = Hopfield(); net.store(P, range(m))
                sw.append(np.mean([np.array_equal(np.where(net.W @ p >= 0, 1, -1), p) for p in P]))
            print(f"{m:>7}{np.mean(sw)*100:>9.0f}%")
        print("\nDigit images are strongly correlated, so their practical capacity is far below the random-pattern limit.")

    # -- 6
    def op6(self):
        self.need()
        n = int(input("Random starting states to probe (default 500): ") or 500)
        cnt = {"stored": 0, "mirror": 0, "spurious": 0}; examples = {}; spur = {}
        for _ in range(n):
            x, _, _ = self.net.recall(self.rng.choice([-1, 1], N), self.rng)
            r = self.net.identify(x); cnt[r[0]] += 1
            if r[0] != "stored":
                key = tuple(x)
                spur[key] = spur.get(key, 0) + 1; examples[key] = (x, r)
        print(f"\nFrom {n} random starts the network settled in:")
        print(f"  stored memories : {cnt['stored']:>4} ({cnt['stored']/n*100:.1f}%)")
        print(f"  mirror images   : {cnt['mirror']:>4} ({cnt['mirror']/n*100:.1f}%)   (-pattern is always an attractor)")
        print(f"  other spurious  : {cnt['spurious']:>4} ({cnt['spurious']/n*100:.1f}%)")
        print(f"  distinct false attractors found: {len(spur)}")
        for key, c in sorted(spur.items(), key=lambda t: -t[1])[:3]:
            x, r = examples[key]
            show(x, f"\nFalse memory reached {c}x - " +
                 (f"inverse of digit {r[1]}" if r[0] == 'mirror' else f"mix, nearest digit {r[1]} ({r[2]} px away)"))
            print(f"   energy {self.net.energy(x):.3f}  (stored patterns: "
                  + ", ".join(f"{d}:{self.net.energy(p):.2f}" for d, p in zip(self.digits, self.net.P)) + ")")
        # also: unstable stored patterns
        bad = [d for d, p in zip(self.digits, self.net.P) if not np.array_equal(np.where(self.net.W @ p >= 0, 1, -1), p)]
        if bad: print(f"\nWARNING: stored digits {bad} are not even fixed points (crosstalk).")

    # -- 7
    def op7(self):
        self.need()
        f = float(input("Noise on real handwritten probes (default 0.1): ") or 0.1)
        D = self.digits; k = len(D)
        for name, noisy_f in (("REAL unseen handwritten samples", 0.0), (f"real samples + {f*100:.0f}% noise", f)):
            C = np.zeros((k, k + 1), int)                         # last col = spurious/mirror
            for ti, d in enumerate(D):
                idx = np.where(self.y == d)[0]
                for i in self.rng.choice(idx, min(40, len(idx)), replace=False):
                    x0 = self.noisy(self.Xb[i], noisy_f) if noisy_f else self.Xb[i]
                    r = self.net.identify(self.net.recall(x0, self.rng)[0])
                    C[ti, D.index(r[1]) if r[0] == "stored" else k] += 1
            print(f"\n{name}\n   (rows = true digit, columns = recalled stored digit; 'sp' = spurious/mirror)")
            print("      " + "".join(f"{d:>5}" for d in D) + f"{'sp':>5}{'acc%':>7}")
            for ti, d in enumerate(D):
                print(f"  {d:>3} " + "".join(f"{C[ti,j]:>5}" for j in range(k + 1)) + f"{C[ti,ti]/C[ti].sum()*100:>7.0f}")
            pairs = sorted(((C[a, b], D[a], D[b]) for a in range(k) for b in range(k) if a != b and C[a, b]), reverse=True)[:3]
            if pairs: print("  Most confused: " + ", ".join(f"{a}->{b} ({n})" for n, a, b in pairs))
        ov = sorted(((int((self.net.P[a] == self.net.P[b]).sum()), D[a], D[b])
                     for a in range(k) for b in range(a + 1, k)), reverse=True)[:3]
        if ov: print("\nStored-pattern pairs with greatest pixel overlap: " + ", ".join(f"{a}/{b} ({n}/64)" for n, a, b in ov))

    # -- 8
    def op8(self):
        self.need()
        print("=" * 62); print("HOPFIELD RECALL PERFORMANCE REPORT"); print("=" * 62)
        print(f"Dataset   : {self.source}")
        print(f"Network   : {N} neurons, Hebbian weights, asynchronous updates")
        print(f"Stored    : digits {self.digits} ({self.mode}, {self.rule} rule); capacity limit ~{0.138*N:.1f} random patterns")
        print(f"Stable stored patterns: {self.stable_count()}/{len(self.digits)}")
        print(f"\n{'noise':>7}{'perfect recall':>16}{'right basin':>13}{'avg sweeps':>12}")
        for f in (0.05, 0.1, 0.2, 0.3, 0.4):
            ok = basin = 0; sw = []; T = 30
            for p, d in zip(self.net.P, self.digits):
                for _ in range(T):
                    x, s, _ = self.net.recall(self.noisy(p, f), self.rng); sw.append(s)
                    r = self.net.identify(x); ok += np.array_equal(x, p)
                    basin += (r[0] == "stored" and r[1] == d)
            n = T * len(self.digits)
            print(f"{f*100:>6.0f}%{ok/n*100:>15.1f}%{basin/n*100:>12.1f}%{np.mean(sw):>12.2f}")
        hit = tot = 0
        for d in self.digits:
            for i in self.rng.choice(np.where(self.y == d)[0], 30, replace=False):
                r = self.net.identify(self.net.recall(self.Xb[i], self.rng)[0]); tot += 1
                hit += (r[0] == "stored" and r[1] == d)
        print(f"\nReal unseen handwritten samples recalled to the right stored digit: {hit/tot*100:.1f}% ({hit}/{tot})")
        cnt = 0
        for _ in range(200):
            r = self.net.identify(self.net.recall(self.rng.choice([-1, 1], N), self.rng)[0]); cnt += r[0] != "stored"
        print(f"Random starts ending in a false memory (spurious or mirror): {cnt/2:.1f}%")


def main():
    X, y, source = load_data(sys.argv[1] if len(sys.argv) > 1 else None)
    print(f"[data] {len(y)} samples loaded.\n")
    app = App(X, y, source)
    ops = {"1": app.op1, "2": app.op2, "3": app.op3, "4": app.op4, "5": app.op5,
           "6": app.op6, "7": app.op7, "8": app.op8}
    while True:
        print("\n===== HOPFIELD DIGIT MEMORY SYSTEM =====")
        print("1. Store Handwritten Digit Patterns")
        print("2. Recall a Noisy Digit Pattern")
        print("3. Missing Pixel Reconstruction Challenge")
        print("4. Digit Corruption Tolerance Test")
        print("5. Hopfield Memory Capacity Challenge")
        print("6. False Memory Detector")
        print("7. Digit Confusion Analyzer")
        print("8. Hopfield Recall Performance Report")
        print("9. Exit")
        try:
            c = input("Choice: ").strip()
        except EOFError:
            break
        if c == "9": print("Goodbye."); break
        if c in ops: print(); ops[c]()
        else: print("Invalid choice.")


if __name__ == "__main__":
    main()


[data] Using scikit-learn bundled UCI optdigits (1797 samples).
[data] 1797 samples loaded.


===== HOPFIELD DIGIT MEMORY SYSTEM =====
1. Store Handwritten Digit Patterns
2. Recall a Noisy Digit Pattern
3. Missing Pixel Reconstruction Challenge
4. Digit Corruption Tolerance Test
5. Hopfield Memory Capacity Challenge
6. False Memory Detector
7. Digit Confusion Analyzer
8. Hopfield Recall Performance Report
9. Exit
Choice: 1

Digits to store (e.g. '2 6 7' or '0 1 2 3 4 5 6 7 8 9', default 2 6 7): 7 8 9
Pattern source: 1) class-average prototype (default)  2) one random sample each: 4
Learning rule: 1) Hebbian (default)  2) pseudo-inverse (comparison, handles correlated digits): 2

Stored 3 patterns by the pseudo-inverse rule  W = P^T (P P^T)^-1 P  (prototype)

Digit 7  (energy -30.500)
   . . . # # # . .
   . . # # # # . .
   . . . . . # . .
   . . . . # # . .
   . . # # # # . .
   . . . # # . . .
   . . . # . . . .
   . . . # . . . .

Digit 8  (energy -30.500)
   . . . # # . . .
   . . 